In [1]:
%%writefile Part1.c

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 1024

int main() {

    int i, j ,k;
    double A[N][N];
    double B[N][N];
    double C[N][N];

    // initialize value for matrices
    for (i = 0; i < N; i++) {
        for (j = 0; j < N; j++) {
            A[i][j] = (double)(i + j);
            B[i][j] = (double)(i - j);
            C[i][j] = 0.0;
        }
    }

    // start timer
    clock_t start = clock();

    for (i = 0; i < N; i++) {
        for (j = 0; j < N; j++) {
            for (k = 0; k < N; k++) {
                C[i][j] += A[i][k] * B[k][j];
            }
        }
    }

    // stop timer
    clock_t end = clock();

    // execution tiime
    double time = ((double)(end - start)) / CLOCKS_PER_SEC;

    printf("Execution time: %f seconds.\n", time);

    return 0;
}

Writing Part1.c


In [2]:
!gcc Part1.c -o Part1

In [3]:
!./Part1

In [ ]:
!nvidia-smi

Tue Jun  2 13:55:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%%writefile Part2.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 1024
#define BLOCK_SIZE 16

__global__ void matrixMulKernel(float *A, float *B, float *C, int n)
{
    int i = blockIdx.y * blockDim.y + threadIdx.y;
    int j = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n && j < n)
    {
        float sum = 0.0f;

        for (int k = 0; k < n; k++)
        {
            sum += A[i * n + k] * B[k * n + j];
        }

        C[i * n + j] = sum;
    }
}

int main()
{
    int size = N * N * sizeof(float);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            h_A[i * N + j] = i + j;
            h_B[i * N + j] = i - j;
        }
    }

    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 block(BLOCK_SIZE, BLOCK_SIZE);

    dim3 grid(
        (N + BLOCK_SIZE - 1) / BLOCK_SIZE,
        (N + BLOCK_SIZE - 1) / BLOCK_SIZE
    );

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    matrixMulKernel<<<grid, block>>>(d_A, d_B, d_C, N);

    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    printf("GPU execution time: %f ms\n", milliseconds);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing Part2.cu


In [6]:
!nvcc Part2.cu -o Part2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [7]:
!./Part2

GPU execution time: 115.568802 ms


In [ ]:
%%writefile Part3.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 1024
#define TILE_SIZE 16

// CUDA Kernel
__global__ void tiledMatrixMul(float *A, float *B, float *C, int n)
{
    __shared__ float tileA[TILE_SIZE][TILE_SIZE];
    __shared__ float tileB[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;

    float sum = 0.0f;

    for (int t = 0; t < n / TILE_SIZE; t++)
    {

        tileA[threadIdx.y][threadIdx.x] =
            A[row * n + (t * TILE_SIZE + threadIdx.x)];

        tileB[threadIdx.y][threadIdx.x] =
            B[(t * TILE_SIZE + threadIdx.y) * n + col];

        __syncthreads();

        for (int k = 0; k < TILE_SIZE; k++)
        {
            sum += tileA[threadIdx.y][k] *
                   tileB[k][threadIdx.x];
        }

        __syncthreads();
    }

    // Write result to global memory
    C[row * n + col] = sum;
}

int main()
{
    int size = N * N * sizeof(float);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            h_A[i * N + j] = i + j;
            h_B[i * N + j] = i - j;
        }
    }

    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 block(TILE_SIZE, TILE_SIZE);

    dim3 grid(
        N / TILE_SIZE,
        N / TILE_SIZE
    );

    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    tiledMatrixMul<<<grid, block>>>(d_A, d_B, d_C, N);

    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float milliseconds = 0;

    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    printf("Tiled GPU execution time: %f ms\n", milliseconds);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing Part3.cu


In [9]:
!nvcc Part3.cu -o Part3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [10]:
!./Part3

Tiled GPU execution time: 20.512480 ms


In [ ]:
%%writefile Part4_NonCoalesced.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 1024
#define BLOCK_SIZE 16
#define STRIDE 32

__global__ void nonCoalescedMatMul(float *A, float *B, float *C, int n)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < n && col < n)
    {
        float sum = 0.0f;

        for(int k = 0; k < n; k++)
        {
            float a = A[row * n + k];

            int bad_col = (col * STRIDE) % n;

            float b = B[k * n + bad_col];

            sum += a * b;
        }

        C[row * n + col] = sum;
    }
}

int main()
{
    int size = N * N * sizeof(float);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for(int i = 0; i < N; i++)
    {
        for(int j = 0; j < N; j++)
        {
            h_A[i * N + j] = i + j;
            h_B[i * N + j] = i - j;
        }
    }

    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 block(BLOCK_SIZE, BLOCK_SIZE);

    dim3 grid(
        (N + BLOCK_SIZE - 1) / BLOCK_SIZE,
        (N + BLOCK_SIZE - 1) / BLOCK_SIZE
    );

    nonCoalescedMatMul<<<grid, block>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    nonCoalescedMatMul<<<grid, block>>>(d_A, d_B, d_C, N);

    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float milliseconds = 0;

    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    printf("Non-coalesced execution time: %f ms\n", milliseconds);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Overwriting Part4_NonCoalesced.cu


In [13]:
!nvcc Part4_NonCoalesced.cu -o Part4_NonCoalesced

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [14]:
!./Part4_NonCoalesced

Non-coalesced execution time: 29.346336 ms


In [ ]:
%%writefile Part4_Coalesced.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 1024
#define BLOCK_SIZE 16

__global__ void coalescedMatMul(float *A,
                                float *B,
                                float *C,
                                int n)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < n && col < n)
    {
        float sum = 0.0f;

        for (int k = 0; k < n; k++)
        {
            // Access matrix A
            float a = A[row * n + k];

            // Coalesced access for B
            float b = B[k * n + col];

            sum += a * b;
        }

        C[row * n + col] = sum;
    }
}

int main()
{
    int size = N * N * sizeof(float);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            h_A[i * N + j] = (float)(i + j);
            h_B[i * N + j] = (float)(i - j);
        }
    }

    float *d_A;
    float *d_B;
    float *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 block(BLOCK_SIZE, BLOCK_SIZE);

    // Grid configuration
    dim3 grid(
        (N + BLOCK_SIZE - 1) / BLOCK_SIZE,
        (N + BLOCK_SIZE - 1) / BLOCK_SIZE
    );

    coalescedMatMul<<<grid, block>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    coalescedMatMul<<<grid, block>>>(d_A, d_B, d_C, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0.0f;

    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess)
    {
        printf("CUDA Error: %s\n",
               cudaGetErrorString(err));
    }

    cudaMemcpy(h_C,
               d_C,
               size,
               cudaMemcpyDeviceToHost);

    printf("Coalesced GPU execution time: %.3f ms\n",
           milliseconds);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing Part4_Coalesced.cu


In [16]:
!nvcc Part4_Coalesced.cu -o Part4_Coalesced

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [17]:
!./Part4_Coalesced

Coalesced GPU execution time: 9.200 ms


In [ ]:
%%writefile Part5.cu

#include <stdio.h>
#include <stdlib.h>

#include <cuda_runtime.h>
#include <cublas_v2.h>

#define N 1024

int main()
{
    int size = N * N * sizeof(float);

    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            h_A[i * N + j] = (float)(i + j);
            h_B[i * N + j] = (float)(i - j);
        }
    }

    float *d_A;
    float *d_B;
    float *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    cublasHandle_t handle;
    cublasCreate(&handle);

    float alpha = 1.0f;
    float beta  = 0.0f;

    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    cublasSgemm(
        handle,
        CUBLAS_OP_N,
        CUBLAS_OP_N,
        N,
        N,
        N,
        &alpha,
        d_B,
        N,
        d_A,
        N,
        &beta,
        d_C,
        N
    );

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0.0f;

    cudaEventElapsedTime(&milliseconds,
                         start,
                         stop);

    cudaMemcpy(h_C,
               d_C,
               size,
               cudaMemcpyDeviceToHost);

    printf("cuBLAS SGEMM time: %.3f ms\n",
           milliseconds);

    cublasDestroy(handle);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing Part5.cu


In [19]:
!nvcc Part5.cu -lcublas -o Part5

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [20]:
!./Part5

cuBLAS SGEMM time: 0.619 ms
